# Marco Geoestadistico


In [2]:
import yarl 
import requests
from scripts.download import download
from scripts.config import DATA_FOLDERS

ext_data = DATA_FOLDERS['external']

inegi_mexico = "https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551163061_s.zip"
inegi_s = "https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551196649/mg_integrado_encuesta_intercensal_2025.zip"

MEXICO_URL = yarl.URL(inegi_mexico)
mexico_map = requests.get(inegi_s)



if mexico_map.status_code:
    download(yarl.URL(inegi_s), "Mexico.zip", ext_data)



https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551196649/mg_integrado_encuesta_intercensal_2025.zip Mexico.zip /home/david/Documents/Repositories/academica/proyecto_agronomia/data/external /home/david/Documents/Repositories/academica/proyecto_agronomia/data/external/Mexico.zip
La conexión procedio. Comenzamos con la descarga
Archivo Mexico.zip descargado correctamente
Descargado el 2026-10-06
Se encuentra en /home/david/Documents/Repositories/academica/proyecto_agronomia/data/external/Mexico.zip


In [6]:
from pathlib import Path

import requests

url = inegi_mexico
output = DATA_FOLDERS['external'] / "Mexico.zip"

chunk_size = 4 * 1024 * 1024  # 4 MB

downloaded = output.stat().st_size if output.exists() else 0

headers = {}

if downloaded:
    headers["Range"] = f"bytes={downloaded}-"

with requests.get(
    url,
    headers=headers,
    stream=True,
    timeout=(10, 120),
) as response:

    # 206 = servidor aceptó Range
    # 200 = descarga completa normal
    if downloaded and response.status_code != 206:
        print("El servidor no soportó reanudación.")
        downloaded = 0
        mode = "wb"
    else:
        mode = "ab" if downloaded else "wb"

    response.raise_for_status()

    remaining = int(response.headers.get("content-length", 0))
    total = downloaded + remaining

    with open(output, mode) as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if not chunk:
                continue

            f.write(chunk)
            downloaded += len(chunk)

            if total:
                print(
                    f"\r{downloaded / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GB "
                    f"({100 * downloaded / total:.1f}%)",
                    end="",
                )

print("\nDescarga terminada.")

2.70 / 2.70 GB (100.0%)
Descarga terminada.


# Censo Agropecuario 2022

In [46]:
from scripts.config import DATA_FOLDERS
import requests
from bs4 import BeautifulSoup
from io import BytesIO
import zipfile 
import yarl
import os

ext_path = DATA_FOLDERS['external']

access_url = yarl.URL("https://www.inegi.org.mx/programas/ca/2022/#datos_abiertos")
down_url = yarl.URL("https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos")
data_year = "ca_2022"

files = ["upaf", "superficie", "upagro", "upfores"]
filatype = "csv"

url_files = [down_url / f"{data_year}_{lf}_csv.zip" for lf in files]
print(url_files)
k = 0
for uf in url_files:
    req = requests.get(uf)
    if not os.path.exists(ext_path / f"file_{k}"):
        os.mkdir(ext_path / f"file_{k}")
    if req.status_code == 200:
        zf = zipfile.ZipFile(BytesIO(req.content))
        zf.extractall(ext_path / f"file_{k}")
    k+=1


[URL('https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_upaf_csv.zip'), URL('https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_superficie_csv.zip'), URL('https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_upagro_csv.zip'), URL('https://www.inegi.org.mx/contenidos/programas/ca/2022/datosabiertos/ca_2022_upfores_csv.zip')]


In [26]:
page_body = soup.find("main")

print(page_body.prettify())

<main>
 <bnavegacion-inegi seccion="programas" titulo="">
 </bnavegacion-inegi>
 <titulo-inegi texto="">
 </titulo-inegi>
 <div class="row">
  <menu-gen class="col-lg-3 col-12" idm="3213" tipo="1" url="">
  </menu-gen>
  <div class="col-12 col-lg-9 ps-lg-0">
   <div class="card">
    <div class="card-body">
     <presentacion-gen>
     </presentacion-gen>
     <relaciontp-gen idc="3213">
     </relaciontp-gen>
     <!--Próximas publicaciones-->
     <div data-esprensa="0" data-idindicador="3213" id="calProy">
     </div>
     <pestanas-gen>
     </pestanas-gen>
    </div>
   </div>
  </div>
 </div>
</main>

